# IADS5 · TP integrador — Ortelana Textil
### Actividad 16 · Serialización y defensa · Entrega final

**Integrantes del grupo:** Camila Belén Capua, Franco Salama, Juan Pablo Ruax Debruijn, Lorena Wajchman, Tomas Rossi y Miguel Ángel Tadakuma  
**Comisión:** [Completar]  
**Repositorio GitHub:** https://github.com/francosalama/paa-integrador-ortelana  
**Fecha:** 17/11/2026

> Notebook independiente. Requiere el archivo `particion_train_test.csv` (partición train/test generada por `actividad_01.ipynb`).

## Actividad 16 · Serialización y defensa · Entrega final

**Qué se espera:** Serializá el pipeline con joblib en `models/modelo_productivo.joblib`, recargalo y comprobá predicciones idénticas; generá requirements.txt y `src/predecir.py` ejecutable; verificá repositorio y prepará defensa de 10–15 minutos.

**Evidencia a entregar:** Repositorio completo, modelo serializado, script, dependencias y defensa oral.

### TODO — Implementación libre
Desarrollá una solución propia. Organizá tantas celdas como necesites; incluí tablas y gráficos con títulos, ejes y unidades cuando correspondan. No alcanza con que el código ejecute: debe permitir comprobar las consignas.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # avisos de versión de librerías
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv("ventas_historicas_ortelana_argentina.csv")

# Partición de la actividad 01: misma semilla, mismas filas de train y test
target = "Format_Reporte"
features = ["Año", "Metros1", "Metros2"]   # sin Sucursal_ID ni Tela_ID (actividad 13)
train, test = train_test_split(df, test_size=0.2, stratify=df[target], random_state=RANDOM_STATE)

# Control: tiene que coincidir con la partición guardada en particion_train_test.csv
particion = pd.read_csv("particion_train_test.csv", index_col="fila")["particion"]
assert (particion[test.index] == "test").all() and (particion == "test").sum() == len(test)

X_train, X_test = train[features], test[features]
y_train, y_test = train[target], test[target]

print("Train:", X_train.shape, "| Test:", X_test.shape)


Train: (2628, 3) | Test: (657, 3)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import joblib
from pathlib import Path

num_cols = ["Año", "Metros1", "Metros2"]

def crear_pipeline_final(modelo):
    """Solo variables de volumen y año: imputar con la mediana (sin escalar, RF es invariante a la escala).
    Sucursal_ID y Tela_ID no entran al modelo: con un par conocido alcanza con su formato histórico."""
    preprocesador = ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median"))]), num_cols),
    ], verbose_feature_names_out=False)
    return Pipeline([("preprocesamiento", preprocesador), ("modelo", modelo)])

# Modelo final de la actividad 15 (configuración del GridSearchCV de la actividad 14)
modelo_final = crear_pipeline_final(RandomForestClassifier(n_estimators=300, max_features=3, min_samples_leaf=3,
                                                       random_state=RANDOM_STATE, n_jobs=-1))
modelo_final.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](4,)","['B','C','M','S']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['Año','Metros1','Metros2']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_na

In [3]:
# 16.1 Serializar el pipeline con joblib, recargarlo y comprobar que predice igual
Path("models").mkdir(exist_ok=True)
joblib.dump(modelo_final, "models/modelo_productivo.joblib")

modelo_cargado = joblib.load("models/modelo_productivo.joblib")
print("Tamaño del archivo:", round(Path("models/modelo_productivo.joblib").stat().st_size / 1e6, 2), "MB")
print("Predicciones idénticas:", (modelo_final.predict(X_test) == modelo_cargado.predict(X_test)).all())
diferencia = np.abs(modelo_final.predict_proba(X_test) - modelo_cargado.predict_proba(X_test)).max()
# Con n_jobs=-1 los árboles suman sus probabilidades en paralelo y el orden de la suma puede variar en el último decimal
print(f"Probabilidades iguales (diferencia máxima {diferencia:.1e}):", diferencia < 1e-12)

Tamaño del archivo: 9.67 MB
Predicciones idénticas: True
Probabilidades iguales (diferencia máxima 2.2e-16): True


In [4]:
# 16.2 requirements.txt con las versiones usadas
import sklearn, xgboost, scipy, matplotlib

requirements = f"""pandas=={pd.__version__}
numpy=={np.__version__}
scikit-learn=={sklearn.__version__}
xgboost=={xgboost.__version__}
scipy=={scipy.__version__}
joblib=={joblib.__version__}
matplotlib=={matplotlib.__version__}
seaborn=={sns.__version__}
"""
with open("requirements.txt", "w") as archivo:
    archivo.write(requirements)
print(requirements)

pandas==3.0.5
numpy==2.5.2
scikit-learn==1.9.0
xgboost==3.4.1
scipy==1.18.1
joblib==1.6.0
matplotlib==3.11.1
seaborn==0.13.2



In [5]:
# 16.3 Script de predicción src/predecir.py
with open("src/predecir.py", encoding="utf-8") as archivo:
    print(archivo.read())

"""Predice Format_Reporte para registros nuevos de Ortelana Textil.

El modelo usa solo Año, Metros1 y Metros2: no usa Sucursal_ID ni Tela_ID. Para un par (Sucursal, Tela) ya
conocido alcanza con su formato histórico (es constante por par); el modelo está pensado para pares nuevos,
donde su F1 macro estimado con validación agrupada por par es ~0,89 (ver actividades 13-15 y README).
Si la entrada trae otras columnas (por ejemplo Sucursal_ID o Tela_ID), se conservan en la salida pero no se usan.

Uso:
    python src/predecir.py --entrada registros.csv [--salida predicciones.csv] [--modelo models/modelo_productivo.joblib]
"""
import argparse
import sys
from pathlib import Path

import joblib
import pandas as pd

RAIZ = Path(__file__).resolve().parents[1]
MODELO_POR_DEFECTO = RAIZ / 'models' / 'modelo_productivo.joblib'
COLUMNAS = ['Año', 'Metros1', 'Metros2']


def main():
    parser = argparse.ArgumentParser(description='Predicción de Format_Reporte (Ortelana Textil)')
    parser.add_arg

In [6]:
# 16.4 Prueba del script desde la línea de comandos (5 registros de test, uno con un NaN)
import subprocess
import sys
import tempfile

ejemplo = test[["Sucursal_ID", "Tela_ID"] + features].head(5).copy()   # el script acepta y conserva columnas extra
ejemplo.loc[ejemplo.index[0], "Metros2"] = np.nan

with tempfile.TemporaryDirectory() as carpeta:   # archivos de prueba temporales: no quedan en el repositorio
    entrada = Path(carpeta) / "entrada.csv"
    salida = Path(carpeta) / "predicciones.csv"
    ejemplo.to_csv(entrada, index=False)
    proceso = subprocess.run([sys.executable, "src/predecir.py", "--entrada", str(entrada), "--salida", str(salida)],
                             capture_output=True, text=True)
    print("Código de salida:", proceso.returncode, "|", (proceso.stdout or proceso.stderr).strip())
    predicciones = pd.read_csv(salida)

display(predicciones)
print("Coinciden con el modelo en memoria:",
      (predicciones["Format_Reporte_predicho"].values == modelo_final.predict(ejemplo[features])).all())

Código de salida: 0 | 5 predicciones guardadas en C:\Users\salam\AppData\Local\Temp\tmp17k_38zt\predicciones.csv


,Sucursal_ID,Tela_ID,Año,Metros1,Metros2,Format_Reporte_predicho,prob_B,prob_C,prob_M,prob_S
0,5,TELA-001,2016,292.35,NaN,C,0.3700,0.5851,0.0,0.0448
1,2,TELA-006,2018,2682.03,2106.97,C,0.0000,0.8988,0.0,0.1012
2,4,TELA-009,2025,1575.90,1150.65,C,0.0000,0.9428,0.0,0.0572
3,5,TELA-006,2023,1114.15,1152.13,S,0.0162,0.0016,0.0,0.9822
4,10,TELA-018,2019,1003.58,773.20,C,0.0000,0.9557,0.0,0.0443


Coinciden con el modelo en memoria: True


In [7]:
# 16.5 Verificación del repositorio
archivos = [f"actividad_{i:02d}.ipynb" for i in range(1, 17)] + [
    "ventas_historicas_ortelana_argentina.csv", "particion_train_test.csv", "README.md",
    "requirements.txt", "models/modelo_productivo.joblib", "src/predecir.py"]
pd.DataFrame({"existe": [Path(a).exists() for a in archivos]}, index=archivos)

,existe
actividad_01.ipynb,True
actividad_02.ipynb,True
actividad_03.ipynb,True
actividad_04.ipynb,True
actividad_05.ipynb,True
actividad_06.ipynb,True
actividad_07.ipynb,True
actividad_08.ipynb,True
actividad_09.ipynb,True
actividad_10.ipynb,True


### Explicación del alumno/a

**Decisiones técnicas y justificación:**
- **Serialización:** guardamos con `joblib` el **pipeline completo** (imputación + Random Forest, sin escalado, solo con `Año`, `Metros1` y `Metros2`) en `models/modelo_productivo.joblib` (~9,7 MB). Así, en producción no hay que repetir ningún paso de preparación a mano.
- **Verificación al recargar:** las predicciones sobre el test son idénticas y las probabilidades coinciden hasta el redondeo de punto flotante (diferencia < 10⁻¹²: con `n_jobs=-1`, los 300 árboles suman en paralelo y el orden de la suma puede variar en el último decimal).
- **`requirements.txt`:** fija las versiones exactas usadas: pandas 3.0.5, numpy 2.5.2, scikit-learn 1.9.0, xgboost 3.4.1, scipy 1.18.1, joblib 1.6.0, matplotlib 3.11.1 y seaborn 0.13.2. Un pipeline serializado debe cargarse con la misma versión de scikit-learn.
- **`src/predecir.py`:** script ejecutable con `argparse` que recibe un CSV, valida que estén `Año`, `Metros1` y `Metros2`, carga el modelo y devuelve la clase predicha y la probabilidad de cada clase. Si el CSV trae otras columnas (por ejemplo, la sucursal y la tela), las conserva en la salida pero no las usa. Lo probamos desde la línea de comandos con 5 registros (uno con NaN): terminó con código 0 y sus predicciones coinciden con las del modelo en memoria.
- **Repositorio:** verificamos que existan los 16 notebooks, el dataset, el README, `requirements.txt`, el modelo, el script y la partición.

**Interpretación de los resultados:**
- **Artefacto productivo:** el modelo final (F1 macro de test 0,9107; ~0,89 estimado para pares nuevos) queda listo como un único archivo reproducible.
- **Validación del script:** confirma que se puede usar fuera del notebook, con registros incompletos, y que entrega probabilidades para identificar predicciones dudosas. Por ejemplo, el registro con NaN en `Metros2` se predice C con P = 0,59, y es un error (su formato real es M): una probabilidad baja es justamente la señal para revisarlo.

**Pregunta de análisis:** ¿Cómo evitaste data leakage? ¿Qué limitaciones tiene el modelo y cómo lo monitorearías?
- **Cómo evitamos la fuga:**
  1. Excluimos `Metros3`–`Metros12`, cuyo patrón de faltantes revela el target.
  2. Separamos el test una sola vez al comienzo y persistimos sus índices.
  3. Ajustamos todo el preprocesamiento (imputadores, escaladores, one-hot) dentro de `Pipeline`, que se reajusta en cada fold.
  4. En la selección del modelo final, validamos con CV agrupada por par, para que ningún par (Sucursal, Tela) esté a la vez en entrenamiento y validación.
  5. Seleccionamos modelos e hiperparámetros solo con CV sobre train.
  6. Analizamos los errores con predicciones fuera de fold.
  7. Hicimos el clustering sin el target.
  8. En la regresión, el target es el volumen anual y excluimos todas las columnas `Metros1`–`Metros12`, porque son sus sumandos.
- **Limitaciones:**
  - El formato es constante para cada par (Sucursal, Tela). Los modelos con identificadores lo memorizaban: 0,97 con la partición aleatoria, pero 0,64 con la CV agrupada por par (Actividad 13). Por eso el modelo final **no usa `Sucursal_ID` ni `Tela_ID`**: para pares conocidos alcanza con el formato histórico, y para pares nuevos el modelo rinde ~0,89 (CV agrupada).
  - Las confusiones se concentran entre S, C y M cuando el volumen es atípico para la clase (S y C tienen la misma cantidad de períodos y volúmenes parecidos).
  - La descripción del dataset explica qué es cada formato, pero no el costo de negocio de confundirlos.
  - Supone que `Metros1` y `Metros2` están disponibles al predecir.
- **Monitoreo:**
  - **Datos:** la tasa de faltantes y la deriva en la distribución de `Año`, `Metros1` y `Metros2` (por ejemplo con PSI o la prueba KS).
  - **Predicciones:** la distribución de las clases predichas y la proporción de predicciones con probabilidad máxima < 0,6.
  - **Desempeño:** cuando llegue la etiqueta real, F1 macro por período contra el ~0,89 de referencia para pares nuevos, con un umbral de alerta (por ejemplo < 0,80) que dispare un reentrenamiento con los datos nuevos.

**Limitaciones o mejoras posibles:**
- El script no valida tipos ni rangos más allá de las columnas.
- Se podría agregar un esquema de validación de la entrada, logging de las predicciones y una consulta previa al histórico: si el par (Sucursal, Tela) ya existe, devolver su formato conocido y usar el modelo solo para pares nuevos.

### Comentarios del docente

[Espacio breve para observaciones sobre esta actividad: aciertos, correcciones o sugerencias.]


---
# Evaluación global del TP integrador

**Rúbrica sugerida para la evaluación global del trabajo** (orientativa; el enunciado remite a una rúbrica final de clase 16 que no está incluida en el documento proporcionado):

| Criterio | Máximo | Puntaje docente |
|---|---:|---:|
| Cumplimiento de consignas y evidencias | 30 | ____ |
| Corrección técnica y ausencia de leakage | 25 | ____ |
| Interpretación y justificación | 25 | ____ |
| Reproducibilidad, claridad y documentación | 20 | ____ |
| **Total** | **100** | **____** |

**Calificación única del TP integrador:** [____ / 10]  
**Peso de la entrega final según el enunciado:** 60% de la calificación de la materia. **La ponderación restante no se define en este cuaderno.**

**Devolución final del docente:**  
[Escribí aquí una devolución global.]

**Preguntas para la defensa oral:**  
[Anotá preguntas y respuestas observadas.]

**Resultado final:** [Aprobado / Reentrega / Pendiente]

**Docente:** [Nombre]  
**Fecha de corrección:** [Fecha]
